# AIE S4 — California Housing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/python-ai-engineering/challenges/aie-s4-california-housing.ipynb)

**Regression.** Predict the median house value of a California census
district from eight numeric features.

Challenge: <https://ml-arena.com/viewchallenge/187>

---

Session 2 ended with a straight line. This notebook replaces it with
a neural network, and the point is not the score — it is that you
should be able to **explain every line by the end**.

So it is built from the bottom up. Nothing appears as a library call
before you have done it by hand:

| section | what you build | what replaces it |
|---|---|---|
| 4 | a linear model as `X @ w + b`, on raw tensors | — |
| 5 | its gradients, from `.backward()`, updated by hand | — |
| 6 | the same thing again | `nn.Linear`, `nn.MSELoss`, `optim.SGD` |
| 7 | a hidden layer | `nn.Sequential` |
| 8 | mini-batches and a validation loop | `DataLoader` |

Section 6 will reproduce section 5's number to four decimals. That
equality is the whole reason for doing it twice.

**Do the two warm-up notebooks first** if you have not: *Optimization
& Training* and *CPU vs GPU*. They need no account.

---

## 0. Setup

`torch`, `pandas`, `scikit-learn` and `matplotlib` all ship with
Colab. The only install is the ML-Arena client, which downloads the
data and uploads your answer.

The distribution is **`mlarena-sdk`** and it imports as `mlarena`. Do
not `pip install mlarena` — that is an unrelated package by another
author, and none of the calls below exist in it.

In [ ]:
!pip install -q mlarena-sdk

---

## 1. Get the data

Paste your personal API key from your ML-Arena **Profile** page. It
starts with `mlk_user_`.

In [ ]:
import mlarena

API_KEY = "mlk_user_..."   # <- paste yours here
CHALLENGE_ID = 187

client = mlarena.connect(api_key=API_KEY)
client.download_dataset(CHALLENGE_ID, ".")

---

## 2. Read it

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

X = pd.read_csv("X.csv")
y = pd.read_csv("y.csv")["prediction"]
X_submission = pd.read_csv("X_submission.csv")

FEATURES = [c for c in X.columns if c != "id"]
print("X", X.shape, " X_submission", X_submission.shape)
print("p =", len(FEATURES), "features:", FEATURES)
X.head()

**n = 16,512 training districts, p = 8 features**, all numeric — so
there is nothing to one-hot encode. That is deliberate: this notebook
is about PyTorch, and every minute spent on encoding is a minute not
spent on the training loop.

The target is the district's median house value **in units of
$100,000**, so `2.5` means $250,000.

In [ ]:
print(y.describe().round(3))
X[FEATURES].describe().round(2)

Look at the `max` row of that second table before going on. `population`
reaches 35,682 and `avg_bedrooms` sits near 1. **Those two columns
differ by four orders of magnitude**, and section 9 measures what that
does to a neural network. Remember that you saw it here.

---

## 3. Look at it

Three plots. Each one changes something you are about to do.

In [ ]:
import seaborn as sns
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(y, bins=60, ax=axes[0], color="#2f6f9f")
axes[0].set_title("Median house value ($100k)")
axes[0].set_xlabel("target")

corr = X[FEATURES].assign(target=y).corr()["target"]
corr = corr.drop("target").sort_values()
corr.plot.barh(ax=axes[1], color="#2f6f9f")
axes[1].set_title("Linear correlation with the target")
plt.tight_layout()
plt.show()

n_capped = int((y >= 5.0).sum())
print(f"{n_capped} of {len(y)} training districts sit at the 5.0 cap")

**The spike at the right edge is real.** The census capped the value
at $500,000, so every district worth more than that is recorded as
exactly 5.0. No model can predict past it, and section 10 uses that
fact to pick up free score.

The correlation bars say `median_income` carries most of the linear
signal and the two coordinates carry almost none. Hold that thought —
it is wrong, and the next plot shows why.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
pts = ax.scatter(X["longitude"], X["latitude"], c=y,
                 cmap="viridis", s=4, alpha=0.6)
plt.colorbar(pts, ax=ax, label="median house value ($100k)")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title("Price is geography — but not linearly")
plt.show()

There is the shape of California, and the expensive districts are two
bright blobs: the Bay Area and Los Angeles.

**This is the plot that justifies the whole session.** Latitude alone
correlates with price at about −0.14 and longitude at about −0.05 —
close to nothing, which is what the bar chart said. But *together*
they locate a district precisely, and location is most of the price.

A linear model computes `w1·latitude + w2·longitude`: a single tilted
plane. It can say "north is dearer" or "west is dearer". It cannot say
"expensive **here** and **here**, cheap in between". A hidden layer
can, because ReLU units can carve the plane into regions.

That is the gap this notebook is about to measure.

---

## 4. From DataFrame to tensor

Three steps, and the order matters.

**Split first.** Hold out a validation set *before* touching the data,
for the reason Session 3 spent an hour on.

**Then scale — fitted on the training part only.** `StandardScaler`
subtracts the mean and divides by the standard deviation, so every
column arrives at the network on a comparable scale. Fitting it on
all the data would leak the validation set's mean into training. It is
a small leak and it is still a leak.

**Then convert.** A tensor is a typed n-dimensional array that also
remembers how it was computed. `float32` is the default everywhere in
deep learning: half the memory of `float64` and enough precision, and
passing `float64` into a `float32` layer is a `RuntimeError` rather
than a silent cast.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_fit, X_val, y_fit, y_val = train_test_split(
    X[FEATURES], y, test_size=0.2, random_state=0)

scaler = StandardScaler().fit(X_fit)          # fitted on the fit part ONLY


def to_tensor(frame):
    return torch.tensor(scaler.transform(frame), dtype=torch.float32)


X_fit_t = to_tensor(X_fit)
X_val_t = to_tensor(X_val)
X_submission_t = to_tensor(X_submission[FEATURES])

# .view(-1, 1) turns a length-n vector into an (n, 1) column. The model
# outputs (n, 1), and MSELoss between (n, 1) and (n,) silently
# broadcasts into an (n, n) matrix — a wrong loss with no error.
y_fit_t = torch.tensor(y_fit.values, dtype=torch.float32).view(-1, 1)
y_val_t = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)

print("X_fit_t ", tuple(X_fit_t.shape), X_fit_t.dtype)
print("y_fit_t ", tuple(y_fit_t.shape), y_fit_t.dtype)
print("X_val_t ", tuple(X_val_t.shape))
print()
print("column means after scaling (should be ~0):", X_fit_t.mean(dim=0).numpy().round(3))
print("column stds  after scaling (should be ~1):", X_fit_t.std(dim=0).numpy().round(3))

The broadcasting warning in that cell is not hypothetical. It is the
single most common silent bug in a first PyTorch regression: the loss
still decreases, the run still finishes, and the model is nonsense.
Check your shapes.

---

## 5. A linear model, with no PyTorch magic at all

Before any layer, build the model out of arithmetic.

A linear model is one matrix multiply: `pred = X @ w + b`, where `X` is
(n, 8), `w` is (8, 1) and `b` is a single number. `@` is matrix
multiplication.

`requires_grad=True` is the whole trick. It tells PyTorch to record
every operation these tensors take part in, so that later,
`loss.backward()` can walk that record backwards and fill in
`w.grad` — the derivative of the loss with respect to every entry of
`w`. You wrote that derivative by hand in warm-up 1. You will not
write another one.

In [ ]:
torch.manual_seed(0)

w = torch.zeros(len(FEATURES), 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

predictions = X_fit_t @ w + b
loss = ((predictions - y_fit_t) ** 2).mean()      # MSE, written out

print("predictions:", tuple(predictions.shape))
print("loss:", round(loss.item(), 4))
print("w.grad before backward():", w.grad)

loss.backward()
print("w.grad after  backward():", w.grad.numpy().ravel().round(3))
print("b.grad after  backward():", b.grad.numpy().round(3))

The gradient is one number per feature: how much the loss would rise
if that weight rose. Now the loop — the same five conceptual steps as
warm-up 1, still with no optimizer object.

`with torch.no_grad():` around the update matters. Without it, the
subtraction is itself recorded as part of the computation graph, and
you end up differentiating your own optimizer.

In [ ]:
torch.manual_seed(0)
w = torch.zeros(len(FEATURES), 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
learning_rate = 0.1

for step in range(300):
    pred = X_fit_t @ w + b                       # 1. forward
    loss = ((pred - y_fit_t) ** 2).mean()        # 2. loss
    loss.backward()                              # 3. gradients
    with torch.no_grad():                        # 4. update, untracked
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        w.grad.zero_()                           # 5. clear for next step
        b.grad.zero_()
    if step % 100 == 0:
        print(f"step {step:3d}: loss = {loss.item():.4f}")

from sklearn.metrics import r2_score
with torch.no_grad():
    manual_r2 = r2_score(y_val, (X_val_t @ w + b).numpy().ravel())
print(f"\nhand-written gradient descent, validation R2 = {manual_r2:.4f}")

Now check it against the library that has been doing this for you
since Session 2:

In [ ]:
from sklearn.linear_model import LinearRegression

sk = LinearRegression().fit(scaler.transform(X_fit), y_fit)
sk_r2 = r2_score(y_val, sk.predict(scaler.transform(X_val)))
print(f"sklearn LinearRegression,    validation R2 = {sk_r2:.4f}")
print(f"your loop,                   validation R2 = {manual_r2:.4f}")
print(f"difference: {abs(sk_r2 - manual_r2):.6f}")

**0.6185 against 0.6186** — the printed difference is 0.00016. That
is the most important cell in this notebook.

`LinearRegression` is not a different kind of object from the thing you
just wrote. Twelve lines of tensor arithmetic reproduce it. Everything
that follows — layers, activations, optimizers — is a change to *what*
gets multiplied, never to this loop.

(The residual 0.00016 is not rounding: sklearn solves the normal
equations exactly while your loop descends towards the same answer
and has not quite arrived. Run 3,000 steps instead of 300 and it
shrinks.)

---

## 6. The same model, written the PyTorch way

Now let the library carry it. Three objects replace the twelve lines:

| yours | PyTorch |
|---|---|
| `w`, `b`, `X @ w + b` | `nn.Linear(8, 1)` |
| `((pred - y) ** 2).mean()` | `nn.MSELoss()` |
| the `with torch.no_grad()` block | `optim.SGD(...).step()` |

`nn.Linear(8, 1)` holds exactly the `w` and `b` you made by hand — you
can print them. `optimizer.zero_grad()` is your `.grad.zero_()` calls.
Nothing new is happening.

In [ ]:
torch.manual_seed(0)

model = nn.Linear(len(FEATURES), 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

print("the parameters the optimizer was handed:")
for pname, param in model.named_parameters():
    print(f"  {pname:8} {tuple(param.shape)}")
print(f"  ({sum(p.numel() for p in model.parameters())} numbers — 8 weights + 1 bias)")

for step in range(300):
    pred = model(X_fit_t)              # 1. forward
    loss = loss_fn(pred, y_fit_t)      # 2. loss
    optimizer.zero_grad()              # 3. clear
    loss.backward()                    # 4. gradients
    optimizer.step()                   # 5. update

model.eval()
with torch.no_grad():
    nn_r2 = r2_score(y_val, model(X_val_t).numpy().ravel())
print(f"\nnn.Linear + SGD, validation R2 = {nn_r2:.4f}")
print(f"your hand-written loop,          {manual_r2:.4f}")
print(f"sklearn,                         {sk_r2:.4f}")

Three routes, one number. **Those five lines are the training loop for
every model in this course**, including the ones with a hundred layers.
From here on only the model changes.

About 0.62 is the ceiling for a linear model on this validation split.
Time to raise it.

---

## 7. Add a hidden layer

`nn.Sequential` chains modules: the output of each is the input of the
next.

```
8 features -> Linear(8, 64) -> ReLU -> Linear(64, 64) -> ReLU -> Linear(64, 1) -> 1 number
```

**The `ReLU`s are the model.** `ReLU(x) = max(0, x)`. Take them out and
the three linear layers collapse algebraically into one linear layer —
you would have 4,801 parameters computing something a 9-parameter model
already computes. The non-linearity between the layers is the only
reason depth means anything.

This is also the answer to the geography plot: each ReLU unit switches
on over part of the input space, and stacking them carves the map into
regions that can have their own price level.

In [ ]:
def build_mlp():
    """8 -> 64 -> 64 -> 1, ReLU between. Fresh weights every call."""
    return nn.Sequential(
        nn.Linear(len(FEATURES), 64),
        nn.ReLU(),
        nn.Linear(64, 64),
        nn.ReLU(),
        nn.Linear(64, 1),
    )


torch.manual_seed(0)
mlp = build_mlp()
print(mlp)
print(f"\nparameters: {sum(p.numel() for p in mlp.parameters()):,}")
print("(8x64 + 64) + (64x64 + 64) + (64x1 + 1) =",
      (8 * 64 + 64) + (64 * 64 + 64) + (64 * 1 + 1))

---

## 8. Mini-batches, and the real training loop

Everything so far computed the loss over all 13,209 rows at once. Two
reasons to stop doing that:

1. **It does not fit** once datasets get large — this one would, a
   million images would not.
2. **It converges more slowly.** One update per pass over the data is
   one update per epoch. With batches of 256 you get 52 updates per
   epoch, each on a noisy estimate of the gradient, and the noise
   itself helps escape bad regions.

`TensorDataset` pairs features with targets; `DataLoader` cuts them into
shuffled batches. `shuffle=True` matters — without it every epoch sees
the same batches in the same order and the noise stops being noise.

Four other things appear here for the first time, and all four are
required:

- **`model.train()` / `model.eval()`** switch training-only behaviour
  (dropout, batch-norm) on and off. This model has neither, so it
  changes nothing *today* — write it anyway, because the day you add
  dropout and forget, your validation number is quietly wrong.
- **`torch.no_grad()`** around evaluation: no graph, less memory, faster.
- **A validation loss computed every epoch**, which is the only thing
  that can tell you when to stop.
- **Keeping the best checkpoint**, because the last epoch is rarely the
  best one.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)
mlp = build_mlp()
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3)

loader = DataLoader(TensorDataset(X_fit_t, y_fit_t),
                    batch_size=256, shuffle=True)
print(f"{len(loader)} batches of up to 256 rows per epoch")

N_EPOCHS = 60
train_losses, val_losses = [], []
best = {"val": float("inf"), "state": None, "epoch": 0}

for epoch in range(N_EPOCHS):
    mlp.train()
    for xb, yb in loader:
        optimizer.zero_grad()
        loss_fn(mlp(xb), yb).backward()
        optimizer.step()

    mlp.eval()
    with torch.no_grad():
        tl = loss_fn(mlp(X_fit_t), y_fit_t).item()
        vl = loss_fn(mlp(X_val_t), y_val_t).item()
    train_losses.append(tl)
    val_losses.append(vl)

    if vl < best["val"]:
        best = {"val": vl,
                "state": {k: v.clone() for k, v in mlp.state_dict().items()},
                "epoch": epoch}

    if (epoch + 1) % 10 == 0:
        print(f"epoch {epoch + 1:3d}  train {tl:.4f}  validation {vl:.4f}")

print(f"\nbest validation loss {best['val']:.4f} at epoch {best['epoch'] + 1}")
mlp.load_state_dict(best["state"])      # restore it — do not ship the last epoch

Adam rather than SGD here. It keeps a running estimate of each
parameter's gradient scale and adapts the step size per parameter,
which on a multi-layer network is worth far more than it was on the
single linear layer. `lr=1e-3` is its usual starting point and a
reasonable default for the rest of your life.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(train_losses, label="train", color="#c1553b")
ax.plot(val_losses, label="validation", color="#2f6f9f")
ax.axvline(best["epoch"], ls="--", c="gray", lw=1,
           label=f"best epoch ({best['epoch'] + 1})")
ax.set_xlabel("epoch"); ax.set_ylabel("MSE")
ax.set_title("The two curves are the diagnostic")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

**How to read this plot** — it is the one you will look at all year.

- Both falling, close together: keep training.
- Both flat: more epochs will not help. Change the model.
- Train still falling, validation flat or rising: **overfitting** —
  Session 3's gap, drawn live. Stop at the minimum of the blue curve.
- Validation *below* train: not a miracle. Usually a leak or a
  train-only regulariser. Go and look.

Here the two stay close and both flatten, which says this model is
capacity-limited rather than overfitting: with 4,801 parameters against
13,209 rows there is not much to memorise with.

---

## 9. Score it, and price the scaler

First the honest comparison on the validation split, then the
experiment that section 2 promised.

In [ ]:
mlp.eval()
with torch.no_grad():
    mlp_r2 = r2_score(y_val, mlp(X_val_t).numpy().ravel())

print(f"linear model  validation R2 = {nn_r2:.4f}")
print(f"MLP           validation R2 = {mlp_r2:.4f}")
print(f"gained: {mlp_r2 - nn_r2:+.4f}")

Now train the **identical** network on **unscaled** inputs. Same
architecture, same optimizer, same 60 epochs, same best-checkpoint
restore. The only difference is the `StandardScaler`.

In [ ]:
torch.manual_seed(0)
X_fit_raw = torch.tensor(X_fit.values, dtype=torch.float32)
X_val_raw = torch.tensor(X_val.values, dtype=torch.float32)

unscaled = build_mlp()
opt_raw = torch.optim.Adam(unscaled.parameters(), lr=1e-3)
raw_loader = DataLoader(TensorDataset(X_fit_raw, y_fit_t),
                        batch_size=256, shuffle=True)
best_raw = float("inf"); best_raw_state = None

for epoch in range(N_EPOCHS):
    unscaled.train()
    for xb, yb in raw_loader:
        opt_raw.zero_grad()
        loss_fn(unscaled(xb), yb).backward()
        opt_raw.step()
    unscaled.eval()
    with torch.no_grad():
        vl = loss_fn(unscaled(X_val_raw), y_val_t).item()
    if vl < best_raw:
        best_raw = vl
        best_raw_state = {k: v.clone() for k, v in unscaled.state_dict().items()}

unscaled.load_state_dict(best_raw_state)
unscaled.eval()
with torch.no_grad():
    raw_r2 = r2_score(y_val, unscaled(X_val_raw).numpy().ravel())

print(f"MLP, standardised inputs   validation R2 = {mlp_r2:.4f}")
print(f"MLP, raw inputs            validation R2 = {raw_r2:.4f}")
print(f"linear model (standardised)              = {nn_r2:.4f}")

**The unscaled network scores below the straight line it was supposed
to replace.** Same code, one missing transform.

Why: the first layer computes `sum(w_j · x_j)`. With `population` in
the tens of thousands and `avg_bedrooms` near 1, that sum — and every
gradient flowing back through it — is dominated by one column. The
other seven barely move. A single learning rate cannot suit both
scales at once.

And nothing warns you. The loss goes down, the run completes, the
number is bad. **Standardising is not optional preprocessing for a
neural network; it is part of the model.**

---

## 10. Predict the submission set, and look at the predictions

Refit nothing — use the restored best checkpoint. Two steps: transform
`X_submission` with the **same scaler** (never a new one), then
look at what comes out *before* writing the file.

In [ ]:
mlp.eval()
with torch.no_grad():
    predictions = mlp(X_submission_t).numpy().ravel()

print(f"min {predictions.min():.3f}   max {predictions.max():.3f}")
out_of_range = ((predictions < 0.15) | (predictions > 5.0)).sum()
print(f"{out_of_range} predictions outside [0.15, 5.0]")

The target cannot leave `[0.15, 5.0]` — that is the census cap from
section 3 — so every prediction outside it is known to be wrong before
anyone scores it. Clipping is one line and it cannot hurt: moving a
prediction to the nearest possible value can only reduce its error.

On the linear benchmark the same clip is worth **+0.028 R²** for no
model change at all. Look at your predictions, not only at your loss.

In [ ]:
clipped = np.clip(predictions, 0.15, 5.0)

submission = pd.DataFrame({"id": X_submission["id"], "prediction": clipped})
submission.to_csv("submission.csv", index=False)

assert len(submission) == len(X_submission)
assert submission["id"].is_unique
assert submission["prediction"].notna().all()
print(f"wrote submission.csv — {len(submission)} rows")
submission.head()

---

## 11. Submit

In [ ]:
result = client.submit(challenge_id=CHALLENGE_ID, files=["submission.csv"],
                       agent_name="mlp-64-64")
print(result)

Scoring takes a moment. Then:

In [ ]:
client.leaderboard(CHALLENGE_ID)

**What to expect.** This notebook scores **R² ≈ 0.776** on the
leaderboard, against the linear benchmark's **0.576**. Across seeds
0, 1 and 2 it lands on 0.776, 0.776 and 0.779, so a much lower number
means something is broken rather than unlucky — check, in this order:

| symptom | cause |
|---|---|
| R² around 0.55 | the scaler was skipped, or refitted on `X_submission` |
| R² around 0.62 | the `ReLU`s are missing — it is still a linear model |
| loss became `nan` | learning rate too high |
| loss never moved | missing `optimizer.zero_grad()`, or `lr` far too small |
| validation ≫ train | a leak: the scaler saw the validation rows |

---

## 12. What you should be able to explain

Go back through and make sure you can say, out loud, what each of
these does. This is the actual deliverable of the session — the
leaderboard position is not.

1. `requires_grad=True` — what it switches on, and when it costs you.
2. `loss.backward()` — what is filled in, and where.
3. `with torch.no_grad():` — the two different jobs it does in this
   notebook (section 5's update, section 8's evaluation).
4. `optimizer.zero_grad()` — what breaks without it, and why it fails
   silently.
5. `.view(-1, 1)` — the bug it prevents.
6. `nn.ReLU()` — why removing it makes the other 4,800 parameters
   pointless.
7. `model.train()` / `model.eval()` — why they are here even though
   this model has no dropout.
8. `DataLoader(..., shuffle=True)` — what shuffling is for.
9. Why the scaler is fitted on `X_fit` and not on `X`.
10. Why the best checkpoint is restored instead of shipping epoch 60.

If any of those is a shrug, the fix is to break that line on purpose
and re-run the cell. That takes thirty seconds and it is worth more
than reading it again.

---

**Next:** *AIE S4 — Forest Cover Type*. Same machinery, a class label
instead of a number, and no code — you write it.